In [ ]:
import torch
import torch.nn as nn
from flask import Flask, render_template, request
from PIL import Image
import torchvision.transforms as transforms
import os

app = Flask(__name__)


class BrainTumorCNN(nn.Module):

    def __init__(self):
        super(BrainTumorCNN, self).__init__()

        self.conv = nn.Sequential(

            nn.Conv2d(3,32,3),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32,64,3),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64,128,3),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128*26*26,128),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128,1),
            nn.Sigmoid()
        )

    def forward(self,x):
        x = self.conv(x)
        x = self.fc(x)
        return x

model = BrainTumorCNN()
model.load_state_dict(torch.load("brain_tumor_model.pth", map_location=torch.device('cpu')))
model.eval()
transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor()
])


@app.route('/')
def home():
    return render_template("index.html")



@app.route('/predict', methods=['POST'])
def predict():

    file = request.files['file']

    filepath = os.path.join("static", file.filename)
    file.save(filepath)

    image = Image.open(filepath).convert("RGB")
    image = transform(image)
    image = image.unsqueeze(0)

    with torch.no_grad():
        output = model(image)

    prediction = output.item()

    prediction = output.item()

    confidence = prediction * 100

    if prediction > 0.5:
        result = "Tumor Detected"
    else:
        result = "No Tumor Detected"
        confidence = (1 - prediction) * 100

    confidence = round(confidence, 2)

    return render_template(
        "index.html",
        prediction=result,
        confidence=confidence,
        img_path=filepath
)



if __name__ == "__main__":
    app.run(debug=True, use_reloader=False)

 * Serving Flask app '__main__'
 * Debug mode: on


 * Running on http://127.0.0.1:5000
Press CTRL+C to quit
127.0.0.1 - - [17/Mar/2026 03:59:32] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [17/Mar/2026 03:59:32] "GET /favicon.ico HTTP/1.1" 404 -
127.0.0.1 - - [17/Mar/2026 04:00:11] "POST /predict HTTP/1.1" 200 -
127.0.0.1 - - [17/Mar/2026 04:00:12] "GET /static/1%20no.jpeg HTTP/1.1" 200 -
127.0.0.1 - - [17/Mar/2026 04:00:37] "POST /predict HTTP/1.1" 200 -
127.0.0.1 - - [17/Mar/2026 04:00:37] "GET /static/Y1.jpg HTTP/1.1" 200 -
127.0.0.1 - - [17/Mar/2026 04:01:18] "POST /predict HTTP/1.1" 200 -
127.0.0.1 - - [17/Mar/2026 04:01:18] "GET /static/Y28.jpg HTTP/1.1" 200 -
127.0.0.1 - - [17/Mar/2026 04:05:21] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [17/Mar/2026 04:05:30] "POST /predict HTTP/1.1" 200 -
127.0.0.1 - - [17/Mar/2026 04:05:30] "GET /static/Y7.jpg HTTP/1.1" 200 -
